# Set Up

In [95]:
!pip list | grep transformers

sentence-transformers                    5.1.2
transformers                             4.57.2


In [96]:
# This cell will authenticate you and mount your Drive in the Colab.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [97]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import matplotlib.pyplot as plt

# Data Exploration


In [98]:
def pandas_table_column_exploration(df,table_name = ''):
    """
    Given a pandas df, does some initial exploration
    Input:
        df: pandas df to be explored.
        table_name (optional): what you call the pandas df being explored
    """

    print("\n---------------------------------------------------")
    print("Exploring Columns for Table:", table_name)
    print("---------------------------------------------------\n")

    # Grab all the nummeric columns
    num_cols = df.select_dtypes(include='number').columns

    # Print out some basic charecteristics for each column
    for column_name in df.columns:
        print("---------------------------------------------------")
        print("Column:", column_name)
        print("---------------------------------------------------")

        print('dtype:', df[column_name].dtype)
        print('Number of values:', df[column_name].count())
        print('Null count:', df[column_name].isnull().sum())
        print('Number of unique values:', df[column_name].nunique())
        print('Mode:',df[column_name].mode().unique()) #.to_string(index=False)

        if column_name in num_cols:
            print('Max:',df[column_name].max())
            print('Min:',df[column_name].min())


            # make plot
            plt.figure()
            plt.tight_layout()
            plt.boxplot(df[column_name])
            plt.show()

        elif df[column_name].dtype == str:
            print('Average length:',sum(df[column_name].apply(len))/df[column_name].count())

## Perenual Data

In [99]:
perenual_data = pd.read_csv('./drive/MyDrive/DATASCI 266/Project/perenual_data.csv')

In [100]:
perenual_data.head()

,Unnamed: 0,id,common_name,scientific_name,other_name,family,hybrid,authority,subspecies,cultivar,...,default_image.license_name,default_image.license_url,default_image.original_url,default_image.regular_url,default_image.medium_url,default_image.small_url,default_image.thumbnail,pruning_count.amount,pruning_count.interval,default_image
0,0,1,European Silver Fir,['Abies alba'],['Common Silver Fir'],Pinaceae,NaN,NaN,NaN,NaN,...,Attribution-ShareAlike 3.0 Unported (CC BY-SA ...,https://creativecommons.org/licenses/by-sa/3.0...,https://perenual.com/storage/species_image/1_a...,https://perenual.com/storage/species_image/1_a...,https://perenual.com/storage/species_image/1_a...,https://perenual.com/storage/species_image/1_a...,https://perenual.com/storage/species_image/1_a...,NaN,NaN,NaN
1,1,2,Pyramidalis Silver Fir,"[""Abies alba 'Pyramidalis'""]",[],Pinaceae,NaN,NaN,NaN,Pyramidalis,...,Attribution-ShareAlike License,https://creativecommons.org/licenses/by-sa/2.0/,https://perenual.com/storage/species_image/2_a...,https://perenual.com/storage/species_image/2_a...,https://perenual.com/storage/species_image/2_a...,https://perenual.com/storage/species_image/2_a...,https://perenual.com/storage/species_image/2_a...,NaN,NaN,NaN
2,2,3,White Fir,['Abies concolor'],"['Silver Fir', 'Concolor Fir', 'Colorado Fir']",Pinaceae,NaN,NaN,NaN,NaN,...,Attribution-ShareAlike License,https://creativecommons.org/licenses/by-sa/2.0/,https://perenual.com/storage/species_image/3_a...,https://perenual.com/storage/species_image/3_a...,https://perenual.com/storage/species_image/3_a...,https://perenual.com/storage/species_image/3_a...,https://perenual.com/storage/species_image/3_a...,NaN,NaN,NaN
3,3,4,Candicans White Fir,"[""Abies concolor 'Candicans'""]","['Silver Fir', 'Concolor Fir', 'Colorado Fir']",Pinaceae,NaN,NaN,NaN,Candicans,...,Attribution-ShareAlike License,https://creativecommons.org/licenses/by-sa/2.0/,https://perenual.com/storage/species_image/4_a...,https://perenual.com/storage/species_image/4_a...,https://perenual.com/storage/species_image/4_a...,https://perenual.com/storage/species_image/4_a...,https://perenual.com/storage/species_image/4_a...,2.0,yearly,NaN
4,4,5,Fraser Fir,['Abies fraseri'],['Southern Fir'],Pinaceae,NaN,NaN,NaN,NaN,...,Attribution License,https://creativecommons.org/licenses/by/2.0/,https://perenual.com/storage/species_image/5_a...,https://perenual.com/storage/species_image/5_a...,https://perenual.com/storage/species_image/5_a...,https://perenual.com/storage/species_image/5_a...,https://perenual.com/storage/species_image/5_a...,NaN,NaN,NaN


In [101]:

# filter only to thet columns helpful to us
perenual_data = perenual_data[['description',
'plant_anatomy',
'sunlight',
'soil',
'leaf',
'flowers',
'fruits',
'watering',
'cycle',
'origin',
'care_level',
'maintenance',
'growth_rate',
'common_name',
'dimensions',
'attracts',
'drought_tolerant',
'salt_tolerant',
'thorny',
'edible_fruit',
'harvest_season',
'medicinal',
'poisonous_to_humans',
'poisonous_to_pets']]


In [102]:
#perenual_data['drought_tolerant']

In [103]:
pandas_table_column_exploration(perenual_data, 'perenual_data')


---------------------------------------------------
Exploring Columns for Table: perenual_data
---------------------------------------------------

---------------------------------------------------
Column: description
---------------------------------------------------
dtype: object
Number of values: 2990
Null count: 10
Number of unique values: 2990
Mode: ["'Paint The Town Red' is an amazing shade of Dianthus that truly lives up to its name. Its impressive bright pink blooms are far less common than the standard pinks, purples and whites, and it's sure to stand out in almost any landscape. It's an incredibly hardy species, blooming from summer to early autumn, surviving in most soils and requiring little maintenance. Its easy to look after and highly tolerant of a wide range of conditions, making it an excellent choice for gardeners of all levels. The heady sweet scent produced by the blooms will also fill a garden with fragrance, creating a welcoming environment to enjoy."
 'A leek

In [109]:
perenual_data.isna().sum().sort_values()

,0
plant_anatomy,0
sunlight,0
soil,0
leaf,0
fruits,0
flowers,0
watering,0
origin,0
dimensions,0
attracts,0
